# NumPy broadcasting

Broadcasting lets NumPy perform element-wise operations on arrays with different shapes, without you writing loops. NumPy treats dimensions of size `1` as if they could stretch to match the other array. This stretching is conceptual: NumPy generally does not create a repeated copy of the data.

## The broadcasting rule

Compare the shapes from **right to left** (the trailing dimensions). At each position, the dimensions are compatible when they are equal, or when either one is `1`. If one shape has fewer dimensions, pad it with `1`s on the **left** for this comparison. If every pair is compatible, the result has the larger size from each dimension pair; otherwise NumPy raises a `ValueError`.


In [ ]:
import numpy as np

## Example 1: `(4, 3) + (3,)`

The first array has 4 rows and 3 columns. The second array is a one-dimensional array with 3 values.

Align dimensions on the right:

| Array | Shape after left-padding | Dimensions |
|---|---:|---|
| `a` | `(4, 3)` | 4 rows, 3 columns |
| `b` | `(1, 3)` | 1 row, 3 columns |

The last dimensions match (`3` and `3`). The first dimensions (`4` and `1`) are compatible because one is `1`. So NumPy behaves **as if** `b` were extended down the rows to shape `(4, 3)`, then adds corresponding elements. The original `b` still has shape `(3,)`.

For `a = [[0, 1, 2], [3, 4, 5], ...]` and `b = [0, 1, 2]`, the same `b` values are applied to every row.


In [ ]:
a = np.arange(12).reshape(4, 3)
b = np.arange(3)

print("a shape:", a.shape)
print("b shape:", b.shape)
print("a + b shape:", (a + b).shape)
print("a + b:\n", a + b)

# Show the conceptual expansion. np.broadcast_to creates a broadcasted view
# for demonstration; ordinary `a + b` does not require you to do this.
print("b viewed as shape (1, 3):\n", b.reshape(1, 3))
print("b broadcast to shape (4, 3):\n", np.broadcast_to(b, a.shape))

## Example 2: adding a scalar

A scalar has shape `()`. It can be broadcast to every element of an array, so `10` is added to all 12 values below.


In [ ]:
values = np.arange(12).reshape(3, 4)
print("values shape:", values.shape)
print("values + 10:\n", values + 10)

## Example 3: `(4, 1) + (3,)`

Left-pad `(3,)` to `(1, 3)`, then compare:

- `(4, 1)` represents 4 rows and 1 column.
- `(1, 3)` represents 1 row and 3 columns.
- The row counts `4` and `1` are compatible; the column counts `1` and `3` are compatible.

The output shape is `(4, 3)`. Each value in the column is combined with all three values in the row.


In [ ]:
column = np.array([[10], [20], [30], [40]])  # shape (4, 1)
row = np.array([1, 2, 3])                     # shape (3,)

print("column shape:", column.shape)
print("row shape:", row.shape)
print("result shape:", (column + row).shape)
print("column + row:\n", column + row)

## Example 4: compatible shapes `(3, 4) + (4,)`

The `(4,)` array aligns with the last dimension of `(3, 4)`. It behaves like a `(1, 4)` row and is applied to each of the 3 rows.


In [ ]:
matrix = np.arange(12).reshape(3, 4)
row_values = np.array([100, 200, 300, 400])
print("result shape:", (matrix + row_values).shape)
print(matrix + row_values)

## Example 5: incompatible shapes `(3, 4) + (3,)`

Compare from the right: `4` and `3` are different, and neither is `1`. These shapes are incompatible, so this operation raises a `ValueError`. A length-3 array does not automatically mean “one value per row”; to add one value to each row, reshape it to `(3, 1)`.


In [ ]:
matrix = np.arange(12).reshape(3, 4)
row_values = np.array([10, 20, 30])

try:
    matrix + row_values
except ValueError as error:
    print(type(error).__name__ + ":", error)

# Reshape to a column so each value is paired with one matrix row.
print("fixed result shape:", (matrix + row_values.reshape(3, 1)).shape)
print(matrix + row_values.reshape(3, 1))

## Quick shape checks

- `(2, 3)` and `(3,)` → compatible → `(2, 3)`
- `(4, 1)` and `(3,)` → compatible → `(4, 3)`
- `(3, 4)` and `(3,)` → incompatible (`4` vs `3`)
- `(3, 4)` and `(3, 1)` → compatible → `(3, 4)`

When in doubt, write the shapes on paper, pad the shorter one with `1`s on the left, and compare from right to left.
